# Binance EMA Bot — Exploration

Use this notebook to fetch data, look at charts, and tune the EMA crossover before running the live bot.

Make sure you've copied `.env.example` to `.env` and added your **testnet** API keys from https://testnet.binance.vision/.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.exchange import get_exchange, is_live
from src.data import fetch_history, save, load
from src.strategy import EmaCrossover, BUY, SELL
from src.backtest import run as run_backtest

print('Mode:', 'LIVE' if is_live() else 'TESTNET')

## 1. Fetch some history

In [ ]:
SYMBOL = 'BTC/USDT'
TIMEFRAME = '15m'   # try '5m', '15m', '1h', '4h'
DAYS = 30

ex = get_exchange()
df = load(SYMBOL, TIMEFRAME)
if df is None or len(df) < 200:
    df = fetch_history(ex, symbol=SYMBOL, timeframe=TIMEFRAME, days=DAYS)
    save(df, SYMBOL, TIMEFRAME)
print(f'{len(df)} candles, {df.index[0]} → {df.index[-1]}')
df.tail()

## 2. Compute signals (with HTF filter)

In [ ]:
strat = EmaCrossover(
    fast=9, slow=21,
    htf_minutes=60, htf_fast=20, htf_slow=50,
    stop_loss_pct=0.0, take_profit_pct=0.0, trailing_stop_pct=0.0,  # set >0 to enable
)
sig = strat.compute(df)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sig.index, sig['close'], label='Close', alpha=0.5)
ax.plot(sig.index, sig['ema_fast'], label=f'EMA({strat.fast})')
ax.plot(sig.index, sig['ema_slow'], label=f'EMA({strat.slow})')

# Shade regions where HTF says "uptrend"
ax.fill_between(sig.index, sig['close'].min(), sig['close'].max(),
                where=sig['htf_uptrend'], color='green', alpha=0.05, label='HTF uptrend')

buys = sig[sig['signal'] == BUY]
sells = sig[sig['signal'] == SELL]
ax.scatter(buys.index, buys['close'], marker='^', color='green', s=80, label='BUY', zorder=5)
ax.scatter(sells.index, sells['close'], marker='v', color='red', s=80, label='SELL', zorder=5)

ax.set_title(f'{SYMBOL} {TIMEFRAME} — EMA crossover with HTF filter')
ax.legend(loc='upper left')
plt.show()

## 3. Backtest one configuration

In [ ]:
result = run_backtest(df, strategy=strat, starting_cash=10_000)
print(f'Final equity:  ${result.final_equity:,.2f}')
print(f'Return:        {result.return_pct:+.2f}%')
print(f'Max drawdown:  {result.max_drawdown_pct:.2f}%')
print(f'Trades:        {result.n_trades}')
print(f'Win rate:      {result.win_rate*100:.1f}%')
print(f'Avg win/loss:  {result.avg_win*100:+.2f}% / {result.avg_loss*100:+.2f}%')
print(f'Exit reasons:  {result.exit_reasons}')
buy_hold = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
print(f'Buy & hold:    {buy_hold:+.2f}%')

result.equity_curve.plot(figsize=(14, 4), title='Equity curve');

## 4. Sweep risk-management variants

**Important finding:** for EMA crossover, the natural exit (opposite crossover) is usually tighter than any reasonable stop-loss. Adding stops often hurts — see for yourself.

In [ ]:
configs = [
    # (name, htf_minutes, sl, tp, trail)
    ('vanilla',                    None, 0.0,  0.0,  0.0),
    ('+ HTF filter',               60,   0.0,  0.0,  0.0),
    ('+ SL 2%',                    None, 0.02, 0.0,  0.0),
    ('+ SL 2% + TP 4%',            None, 0.02, 0.04, 0.0),
    ('+ Trail 1.5%',               None, 0.0,  0.0,  0.015),
    ('HTF + SL 2% + TP 4%',        60,   0.02, 0.04, 0.0),
    ('HTF + Trail 2%',             60,   0.0,  0.0,  0.02),
]

rows = []
for name, htf_m, sl, tp, trail in configs:
    s = EmaCrossover(fast=9, slow=21, htf_minutes=htf_m, htf_fast=20, htf_slow=50,
                     stop_loss_pct=sl, take_profit_pct=tp, trailing_stop_pct=trail)
    r = run_backtest(df, strategy=s)
    rows.append({'config': name, 'return%': round(r.return_pct, 1), 'trades': r.n_trades,
                 'win%': round(r.win_rate*100, 1), 'max_dd%': round(r.max_drawdown_pct, 1)})

buy_hold = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
print(f'Buy & hold benchmark: {buy_hold:+.2f}%\n')
pd.DataFrame(rows)

## 5. Sweep EMA periods

In [ ]:
rows = []
for fast in [5, 9, 12, 20]:
    for slow in [21, 50, 100]:
        if fast >= slow:
            continue
        s = EmaCrossover(fast=fast, slow=slow, htf_minutes=60, htf_fast=20, htf_slow=50)
        r = run_backtest(df, strategy=s)
        rows.append({'fast': fast, 'slow': slow, 'return%': round(r.return_pct, 1),
                     'trades': r.n_trades, 'win%': round(r.win_rate*100, 1)})
pd.DataFrame(rows).sort_values('return%', ascending=False)